# import, settings, file structure

In [ ]:
from copy import deepcopy
import os

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import fastplotlib as fpl

import mesmerize_core as mc
import mesmerize_viz

In [ ]:
# also need to install ipywidgets and ipympl
%matplotlib widget

# This is just a pandas table display formatting option
pd.options.display.max_colwidth = 120

# paths

In [ ]:
tsu_type = "cab-" # "org-" # 

In [ ]:
from pathlib import Path

main_path = Path("/Users/zzhao89/Documents/Git_Code/organoid/")
for _output_dir in ("mesmerize-batch", "cnmf"):
  (main_path / _output_dir).mkdir(parents=True, exist_ok=True)

mc.set_parent_raw_data_path(main_path)

batch_path = mc.get_parent_raw_data_path().joinpath(
  "mesmerize-batch/" + tsu_type + "cnmf" + ".pickle")


In [ ]:
df = mc.load_batch(batch_path)

# df = mc.create_batch(batch_path)

# for idx, row in df.iterrows():
#   if row["algo"] == "cnmf" or row["algo"] == "cnmfe":
#     df.caiman.remove_item(row.uuid)


df

# cnmf

In [ ]:
import tifffile

movie_path_s = [
  mc.get_parent_raw_data_path().joinpath(p.relative_to(main_path))
  for p in main_path.joinpath("mcorr/").iterdir() if p.name.startswith(tsu_type)] # "org-"
movie_n = len(movie_path_s)

In [ ]:
# movie_s = [
#   tifffile.TiffFile(str(movie_path)).asarray() for movie_path in movie_path_s]

# %matplotlib widget
# import matplotlib.pyplot as plt
# import ipywidgets as widgets

# fig, ax = plt.subplots(figsize = (10,2))

# def update(idx):
#     ax.clear()
#     ax.plot(movie_s[idx][0,:10].flatten())
#     fig.canvas.draw_idle()

# slider = widgets.IntSlider(
#     value=0, min=0, max=movie_n-1, step=1,
#     description="plot", continuous_update=True
# )
# widgets.interact(update, idx=slider)

## cnmf

In [ ]:
cnmf_param_s = {
  'main': {
    'fr': 1/0.264, # framerate, very important!
    'p': 1,
    'nb': 2,
    'merge_thr': 0.9,
    'rf': 70,
    'stride': 4, # "stride" for cnmf, "strides" for mcorr
    'K': 6,
    'gSig': [5, 5],
    # 'ssub': 1,
    # 'tsub': 1,
    'method_init': 'greedy_roi',
    'min_SNR': 0.3, # the main criterion for rejection
    'rval_thr': 0.8,
    'use_cnn': False, # True doesn't work with updated packages on mac
    # 'min_cnn_thr': 0.8,
    # 'cnn_lowest': 0.1,
    'decay_time': 3},
    'refit': True, # If `True`, run a second iteration of CNMF
}

for movie_idx, movie_path in enumerate(movie_path_s): 
  new_param_s = deepcopy(cnmf_param_s)
  # add param variant to the batch
  df.caiman.add_item(
    algo="cnmf",
    item_name=movie_path.stem, input_movie_path=movie_path,
    params=new_param_s)

# # some variants
# for kk in [4, 5, 6]: 
#   for rf in [140, 160]:
#     for merge_thr in [0.8, 0.9]:
#       new_param_s = deepcopy(cnmf_param_s)
#       # assign the "max_shifts"
#       new_param_s["main"]["rf"] = rf
#       new_param_s["main"]["K"] = kk
#       new_param_s["main"]["merge_thr"] = merge_thr
#       # add param variant to the batch
#       df.caiman.add_item(
#         algo="cnmf",
#         item_name=mcorr_item_name, input_movie_path=mcorr_path,
#         params=new_param_s)

# cnmf_param_diff = df.caiman.get_params_diffs(
#   algo="cnmf", item_name=movie_path.stem#df.iloc[0]["item_name"]
#   )
# cnmf_param_diff
df

## cnmfe

In [ ]:
# gSig_temp = 6
# tsub = 5#1#

# cnmfe_param_s = {
#   "main": {
#     'method_init': 'corr_pnr',  # use this for 1 photon
#     'K': None,
#     'gSig': (gSig_temp, gSig_temp),
#     'gSiz': (4 * gSig_temp + 1, 4 * gSig_temp + 1),
#     'merge_thr': 0.7,
#     'p': 1,
#     'tsub': tsub,
#     'ssub': 1,
#     'rf': 70,
#     'stride': 20,
#     'only_init': True,    # set it to True to run CNMF-E
#     'nb': 0,
#     'nb_patch': 0,
#     'method_deconvolution': 'oasis',       # could use 'cvxpy' alternatively
#     'low_rank_background': None,
#     'update_background_components': True,  # sometimes setting to False improve the results
#     'normalize_init': False,               # just leave as is
#     'center_psf': True,                    # leave as is for 1 photon
#     'ssub_B': 2,
#     'ring_size_factor': 1.4,
#     'del_duplicates': True,                # whether to remove duplicates from initialization
#     'min_corr': 0.6, # rval_thr?
#     'min_pnr': pnr_s[0],  # for constant high activity?
#     }
#   }


# # some variants
# for movie_idx, movie_path in enumerate(movie_path_s): 
#   new_param_s = deepcopy(cnmfe_param_s)
#   # assign the "max_shifts"
#   new_param_s["main"]["pnr"] = pnr_s[movie_idx]
#   # add param variant to the batch
#   df.caiman.add_item(
#     algo="cnmfe",
#     item_name=movie_path.stem, input_movie_path=movie_path,
#     params=new_param_s)


# # cnmfe_param_diff = df.caiman.get_params_diffs(
# #   algo="cnmfe", item_name=movie_path.stem#df.iloc[0]["item_name"]
# #   )
# # cnmfe_param_diff
# df

## run

In [ ]:
for idx, row in df.iterrows():
  if row["outputs"] is not None:
    continue # skip if item has already been run
  process = row.caiman.run()
  # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
  # this is unnecessary on Linux & Mac
  # "DummyProcess" is used for local backend so this is automatic
  if process.__class__.__name__ == "DummyProcess":
    df = df.caiman.reload_from_disk()

In [ ]:
# df modified on disk, reload
df = df.caiman.reload_from_disk()
df

# plot

In [ ]:
movie_name_s = [df.iloc[idx].item_name for idx in range(movie_n)]
movie_s = [df.iloc[idx].caiman.get_input_movie() for idx in range(movie_n)]

spat_s, contour_s, com_s, traj_s = [[0 for g_b in range(2)] for dat_typ in range(4)]

for g_b, g_b_str in zip(range(2), ["good", "bad"]):
  traj_s[g_b] = [
    df.iloc[idx].cnmf.get_temporal(g_b_str) 
    for idx in range(movie_n)]
  spat_s[g_b] = [
    df.iloc[idx].cnmf.get_rcm(g_b_str).spatial.toarray() 
    if traj_s[g_b][idx].shape[0]!=0 
    else np.zeros(
      df.iloc[idx].cnmf.get_rcm("good").spatial.toarray().shape[:1]+(0,))
    for idx in range(movie_n)]
  contour_s[g_b] = [
    df.iloc[idx].cnmf.get_contours(g_b_str, swap_dim=False)[0] 
    for idx in range(movie_n)]
  com_s[g_b] = [
    np.array(df.iloc[idx].cnmf.get_contours(g_b_str, swap_dim=False)[1]) 
    for idx in range(movie_n)]
  for idx, spat in enumerate(spat_s[g_b]):
    shape = spat.shape
    spat_s[g_b][idx] = spat.reshape(
      movie_s[idx].shape[1:][::-1] + shape[-1:]).T

# deal with empty lists (no bad neurons)
for idx in range(movie_n):
  if com_s[1][idx].shape==(0,):
    com_s[1][idx] = np.zeros((0,2))

for idx in range(movie_n):
  print(com_s[0][idx].shape[0], com_s[1][idx].shape[0])

### view all

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets

fig, ax_s = plt.subplots(1, 3, figsize = (9,3))

def update(idx):
  for plt_idx in range(3):
    ax_s[plt_idx].clear()
  ax_s[0].set_title(movie_name_s[idx])
  ax_s[0].imshow(np.std(movie_s[idx], axis = 0))
  for g_b, color in zip(range(2), ['Blues','Reds']):
    ax_s[0].scatter(
      *com_s[g_b][idx].T, color = plt.get_cmap(color)(0.8),
      marker = "o", facecolors="none")
  
  for g_b, color in zip(range(2), ['Blues','Reds']):
    ax_s[g_b+1].imshow(
      np.sum(spat_s[g_b][idx], axis = 0), cmap = color)
  

  #   for contour in contour_s[g_b][idx]:
  #     ax_s[1].plot(
  #       *contour.T, color = color)
    
    fig.canvas.draw_idle()

slider = widgets.IntSlider(
    value=0, min=0, max=movie_n-1, step=1,
    description="plot", continuous_update=True
)
widgets.interact(update, idx=slider)

In [ ]:
cnmf_viz = df.cnmf.viz(
  image_data_options=["input", "rcm"], # cnmfe does not support rcb and residuals 
  # image_data_options=["input", "rcm", "rcb", "residuals"], # cnmf does 
  # image_widget_kwargs={"grid_plot_kwargs": {"size": (1000, 500)}} 
)

# cnmf_viz.show(sidecar=True)
cnmf_viz.show()

In [ ]:
cnmf_viz.close()

### view one by one

In [ ]:
index = 0
current_row = df.iloc[index]

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets

fig, ax = plt.subplots(figsize=(4,4))

def basic_update(idx, g_b, label):
    ax.clear()
    current = spat_s[g_b][index][idx]
    ax.set_title(f"{label} {idx}, nonzeros={np.sum(current > 0)}")
    ax.imshow(current)
    ax.plot(*contour_s[g_b][index][idx].T)
    fig.canvas.draw_idle()

slider_good = widgets.IntSlider(
    value=0, min=0, max=spat_s[0][index].shape[0]-1, step=1,
    description="good", continuous_update=True
)
slider_bad = widgets.IntSlider(
    value=0, min=0, max=spat_s[1][index].shape[0]-1, step=1,
    description="bad", continuous_update=True
)

widgets.interact(
  lambda idx: basic_update(idx, g_b=0, label="good"), idx=slider_good)
widgets.interact(
  lambda idx: basic_update(idx, g_b=1, label="bad"), idx=slider_bad)

In [ ]:
input_movie = current_row.caiman.get_input_movie()
rcm = current_row.cnmf.get_rcm()
# rcb = current_row.cnmf.get_rcb()
# res = current_row.cnmf.get_residuals()

cnmf_iw = fpl.ImageWidget(
  # data=[input_movie, rcm, rcb, res], # cnmf
  # names=["input", "parts", "bg", "res"],
  data=[input_movie, rcm], # cnmfe
  names=["input", "parts"],
  grid_plot_kwargs={"size": (800, 600)}, 
  cmap="gnuplot2"
)
cnmf_iw.show()

In [ ]:
cnmf_iw.close()

In [ ]:
rcm_good = current_row.cnmf.get_rcm("good")
rcm_bad = current_row.cnmf.get_rcm("bad")

cnmf_iw_good = fpl.ImageWidget(
  # data=[rcm_good.max_image, rcm_bad.max_image], # max
  # names=["good_max", "bad_max"],
  data=[rcm_good, rcm_bad], # movie
  names=["good", "bad"],
  grid_plot_kwargs={"size": (900, 450)},
  cmap="gnuplot2"
)
cnmf_iw_good.show()

In [ ]:
cnmf_iw_good.close()

# reject by size and separate clusters

In [ ]:
# assign -2 to bad
cluster_id_s = [
  [0 for idx in range(movie_n)],
  [np.full(len(com_s[1][idx]), -2) for idx in range(movie_n)]]

# for good, assign -1 if too small, otherwise assign 1
for idx in range(movie_n):
  com = com_s[0][idx] # could use com to define additional clusters
  cluster_id = np.zeros(len(com)).astype(int)
  
  too_small = (160 > np.sum(
    spat_s[0][idx]>0, axis = (1, 2)))

  cluster_id[too_small] = -1
  cluster_id[~too_small] = 1
  
  cluster_id_s[0][idx] = cluster_id

In [ ]:
spat_all = [
  np.concatenate([spat_s[g_b][idx] for g_b in range(2)], axis = 0) 
  for idx in range(movie_n)]
contour_all = [
  contour_s[0][idx] + contour_s[1][idx] 
  for idx in range(movie_n)]
com_all = [
  np.concatenate([com_s[g_b][idx] for g_b in range(2)], axis = 0) 
  for idx in range(movie_n)]
traj_all = [
  np.concatenate([traj_s[g_b][idx] for g_b in range(2)], axis = 0) 
  for idx in range(movie_n)]
cluster_id_all = [
  np.concatenate([cluster_id_s[g_b][idx] for g_b in range(2)]) 
  for idx in range(movie_n)]

# backgound
bg_xt_all = [
  [getattr(df.iloc[idx].cnmf.get_rcb(), attr)
   for idx in range(movie_n)] 
  for x_t, attr in enumerate(["spatial", "temporal"])]
for idx in range(movie_n):
  old_arr = bg_xt_all[0][idx]
  shape = old_arr.shape
  bg_xt_all[0][idx] = old_arr.reshape(
    movie_s[idx].shape[1:][::-1] + shape[-1:]).T

# print component num
for idx in range(movie_n):
  print(movie_name_s[idx][4:12], ":",
        np.sum((cluster_id_all[idx]==-2).astype(int)), 
        np.sum((cluster_id_all[idx]==-1).astype(int)),
        "|",
        np.sum((cluster_id_all[idx]==0).astype(int)),
        np.sum((cluster_id_all[idx]==1).astype(int)),
        np.sum((cluster_id_all[idx]==2).astype(int)))

# save

In [ ]:
for idx in range(movie_n):
  np.savez(
    mc.get_parent_raw_data_path().joinpath("cnmf/" + movie_name_s[idx] + ".npz"), 
    traj_all[idx], cluster_id_all[idx], com_all[idx], spat_all[idx],
    bg_xt_all[0][idx], bg_xt_all[1][idx])

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as widgets

fig, ax_s = plt.subplots(2, 3, figsize = (9, 6))
ax_s = ax_s.flatten()

def update(idx):
  for plt_idx in range(5):
    ax_s[plt_idx].clear()
  ax_s[0].set_title(movie_name_s[idx])
  ax_s[0].imshow(np.std(movie_s[idx], axis = 0))
  for cluster_id, color in zip(
    np.arange(-2,3), ["Reds", "Purples","Oranges", "Blues", "Greens"]):
    ax_s[0].scatter(
      *com_all[idx][cluster_id_all[idx]==cluster_id].T, color = plt.get_cmap(color)(0.8),
      marker = "o", facecolors="none")
    ax_s[cluster_id+3].imshow(
      np.sum(spat_all[idx][cluster_id_all[idx]==cluster_id], axis = 0),
      cmap = color)
    
  #   for contour in contour_s[g_b][idx]:
  #     ax_s[1].plot(
  #       *contour.T, color = color)
    
    fig.canvas.draw_idle()

slider = widgets.IntSlider(
    value=0, min=0, max=movie_n-1, step=1,
    description="plot", continuous_update=True
)
widgets.interact(update, idx=slider)